In [1]:
import os
data_dir = '/home/devang/Projects/PilotCrew/Data-Science-Bench/tasks/task_03/data'
print(os.listdir(data_dir))

['admissions.csv', 'meds.csv', 'patients.csv', 'labs.csv']


In [2]:
import pandas as pd
admissions = pd.read_csv(os.path.join(data_dir, 'admissions.csv'))
labs = pd.read_csv(os.path.join(data_dir, 'labs.csv'))
meds = pd.read_csv(os.path.join(data_dir, 'meds.csv'))

print("Admissions:")
print(admissions.head())
print("\nLabs:")
print(labs.head())
print("\nMeds:")
print(meds.head())

Admissions:
  admission_id patient_id  admit_date discharge_date  diagnosis ward
0         A001        P01  2024-02-04     2024-02-08     Sepsis    A
1         A002        P02  2024-02-18     2024-02-19     Asthma    A
2         A003        P03  2024-02-10     2024-02-12     Asthma    C
3         A004        P03  2024-02-21     2024-02-24  Pneumonia    B
4         A005        P04  2024-02-13     2024-02-17  Pneumonia    A

Labs:
  admission_id    lab_date  sodium  creatinine
0         A001  2024-02-04     135         1.0
1         A001  2024-02-05     137         1.1
2         A001  2024-02-06     139         1.2
3         A001  2024-02-07     133         1.3
4         A002  2024-02-18     137         1.3

Meds:
  admission_id med_code  start_day
0         A001      M03          0
1         A001      M04          1
2         A002      M01          0
3         A002      M03          0
4         A003      M03          0


In [3]:
# 1. Find admissions with at least one sodium result < 135
low_sodium_labs = labs[labs['sodium'] < 135].copy()

# 2. Find the date of their first below-135 sodium result
first_low_sodium = low_sodium_labs.groupby('admission_id')['lab_date'].min().reset_index()
first_low_sodium.rename(columns={'lab_date': 'first_low_sodium_date'}, inplace=True)
first_low_sodium['first_low_sodium_date'] = pd.to_datetime(first_low_sodium['first_low_sodium_date'])

# Merge with admissions to get admit_date
admissions['admit_date'] = pd.to_datetime(admissions['admit_date'])
first_low_sodium = first_low_sodium.merge(admissions[['admission_id', 'admit_date']], on='admission_id')

# 3. Find if they had medication code M03
m03_meds = meds[meds['med_code'] == 'M03'].copy()

# Merge with first_low_sodium
merged = first_low_sodium.merge(m03_meds, on='admission_id', how='left')

# 4. Calculate the start date of M03
merged['m03_start_date'] = merged['admit_date'] + pd.to_timedelta(merged['start_day'], unit='D')

# 5. Check if M03 start date is <= first below-135 sodium result
# We need to group by admission_id because an admission might have multiple M03 records
# We want to know if *any* M03 started on or before the first low sodium date
merged['m03_before_or_on_low_sodium'] = merged['m03_start_date'] <= merged['first_low_sodium_date']

# Aggregate by admission_id
result_df = merged.groupby('admission_id')['m03_before_or_on_low_sodium'].any().reset_index()

# 6. Calculate percentage
percentage = (result_df['m03_before_or_on_low_sodium'].sum() / len(result_df)) * 100

# 7. Round to 1 decimal place
print(round(percentage, 1))

35.7


/home/devang/Projects/PilotCrew/Data-Science-Bench/.venv/lib/python3.13/site-packages/pandas/core/arrays/timedeltas.py:1163: RuntimeWarning: invalid value encountered in cast
  int_data = data.astype(np.int64)
